In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from pathlib import Path
import glob

# --- Configuration ---
RESULTS_DIR = "/data/cpanourg/2-hdvc/results/relerr/"
OUTPUT_DIR = Path("/home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

# --- Style (paper-quality minimalist) ---
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 8,
    "font.family": "serif",
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Find all CSV files ---
csv_files = glob.glob(f"{RESULTS_DIR}/deep_*_adc_vs_exact_eval.csv")
methods = {}
for csv_file in csv_files:
    method_name = Path(csv_file).stem.replace("deep_", "").replace("_adc_vs_exact_eval", "")
    methods[method_name] = csv_file

print(f"Found methods: {list(methods.keys())}")

# --- Define x-axis variables for each method ---
x_axis_configs = {
    "PQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "OPQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "LSQpp": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "RaBitQ": ["train_size", "bits_per_dim", "bits_per_vector"],
    "VAQ": ["train_size", "n_subquantizers", "bits_per_vector", "min_bits", "max_bits"],
}

# --- X-axis label mapping ---
x_labels = {
    "train_size": "Training size",
    "n_subquantizers": "Number of subquantizers",
    "nbits": "Bits per subspace",
    "bits_per_vector": "Bits per vector",
    "bits_per_dim": "Bits per dimension",
    "min_bits": "Minimum bits per subspace",
    "max_bits": "Maximum bits per subspace",
}

# --- Function to perform significance test ---
def perform_significance_test(x, y):
    """
    Perform Pearson correlation test and return p-value and sample size.
    
    Pearson correlation is valid for testing linear relationships between continuous variables.
    It tests H0: correlation = 0 (no linear relationship) vs H1: correlation != 0.
    Returns p-value < 0.05 indicates significant linear relationship.
    """
    # Remove NaN values
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]
    
    if len(x_clean) < 3:
        return None, None
    
    # Check if either variable is constant (correlation undefined)
    if np.std(x_clean) == 0 or np.std(y_clean) == 0:
        return None, None
    
    # Pearson correlation test
    try:
        corr, p_value = stats.pearsonr(x_clean, y_clean)
        return p_value, len(x_clean)
    except:
        return None, None

# --- Generate plots for each method ---
for method_name, csv_file in methods.items():
    print(f"\nProcessing {method_name}...")
    df = pd.read_csv(csv_file)
    
    # Get x-axis variables for this method
    method_x_vars = x_axis_configs.get(method_name, ["train_size", "bits_per_vector"])
    
    # Filter to only variables that exist in the dataframe
    available_x_vars = [x for x in method_x_vars if x in df.columns]
    
    for x_var in available_x_vars:
        # Skip if no valid data
        if df[x_var].isna().all():
            continue
        
        # Remove rows with NaN in x or y variables
        df_clean = df[[x_var, "rel_error_mean", "rel_error_std"]].dropna()
        
        if len(df_clean) < 2:
            continue
        
        # --- Plot 1: Mean Relative Error ---
        fig, ax = plt.subplots(figsize=(3.3, 2.2))
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_mean"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight best configuration
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_mean"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_mean"])
        
        # Add significance test result to plot (top-right corner to avoid overlapping points)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=7, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            ax.set_xticks(sorted(df_clean[x_var].unique()))
        
        plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = f"{x_var}_{method_name}_mean_rel_error.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
        
        # --- Plot 2: Standard Deviation of Relative Error ---
        fig, ax = plt.subplots(figsize=(3.3, 2.2))
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_std"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight configuration with minimum std (associated with best mean)
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_std"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_std"])
        
        # Add significance test result to plot (top-right corner to avoid overlapping points)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=7, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        ax.set_ylabel("Std dev of relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            ax.set_xticks(sorted(df_clean[x_var].unique()))
        
        plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = f"{x_var}_{method_name}_std_rel_error.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()

print("\n✅ All plots generated successfully!")


Found methods: ['PQ']

Processing PQ...
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/train_size_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/train_size_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/n_subquantizers_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/n_subquantizers_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/nbits_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/nbits_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/bits_per_vector_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/bits_per_vector

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import glob
from scipy import stats

# --- Configuration ---
RESULTS_DIR = "/data/cpanourg/2-hdvc/results/relerr/hp_tests/first_hp_test/"
OUTPUT_DIR = Path("/home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Style (paper-quality minimalist) ---
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 8,
    "font.family": "serif",
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Find all CSV files ---
csv_files = glob.glob(f"{RESULTS_DIR}/deep_*_adc_vs_exact_eval.csv")
methods_data = {}
for csv_file in csv_files:
    method_name = Path(csv_file).stem.replace("deep_", "").replace("_adc_vs_exact_eval", "")
    methods_data[method_name] = pd.read_csv(csv_file)

print(f"Found methods: {list(methods_data.keys())}")

# --- Define x-axis variables for each method ---
x_axis_configs = {
    "PQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "OPQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "LSQpp": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "RaBitQ": ["train_size", "bits_per_dim", "bits_per_vector"],
    "VAQ": ["train_size", "n_subquantizers", "bits_per_vector", "min_bits", "max_bits"],
}

# --- X-axis label mapping ---
x_labels = {
    "train_size": "Training size",
    "n_subquantizers": "Number of subquantizers",
    "nbits": "Bits per subspace",
    "bits_per_vector": "Bits per vector",
    "bits_per_dim": "Bits per dimension",
    "min_bits": "Minimum bits per subspace",
    "max_bits": "Maximum bits per subspace",
}

# --- Function to perform significance test ---
def perform_significance_test(x, y):
    """
    Perform Pearson correlation test and return p-value and sample size.
    
    Pearson correlation is valid for testing linear relationships between continuous variables.
    It tests H0: correlation = 0 (no linear relationship) vs H1: correlation != 0.
    Returns p-value < 0.05 indicates significant linear relationship.
    """
    # Remove NaN values
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]
    
    if len(x_clean) < 3:
        return None, None
    
    # Check if either variable is constant (correlation undefined)
    if np.std(x_clean) == 0 or np.std(y_clean) == 0:
        return None, None
    
    # Pearson correlation test
    try:
        corr, p_value = stats.pearsonr(x_clean, y_clean)
        return p_value, len(x_clean)
    except:
        return None, None

# --- Find hyperparameters that exist in multiple methods ---
# Collect all hyperparameters and which methods have them
hyperparam_methods = {}
for method_name, df in methods_data.items():
    method_x_vars = x_axis_configs.get(method_name, [])
    for x_var in method_x_vars:
        if x_var in df.columns and not df[x_var].isna().all():
            if x_var not in hyperparam_methods:
                hyperparam_methods[x_var] = []
            hyperparam_methods[x_var].append(method_name)

# Only keep hyperparameters that exist in at least 2 methods
hyperparam_methods = {k: v for k, v in hyperparam_methods.items() if len(v) >= 2}

print(f"\nHyperparameters with multiple methods: {list(hyperparam_methods.keys())}")

# --- Generate combined figures for each hyperparameter ---
for x_var, method_list in hyperparam_methods.items():
    print(f"\nProcessing {x_var} for methods: {method_list}")
    
    # Collect all data for this hyperparameter
    all_data = {}
    x_min, x_max = np.inf, -np.inf
    y_mean_min, y_mean_max = np.inf, -np.inf
    y_std_min, y_std_max = np.inf, -np.inf
    
    for method_name in method_list:
        df = methods_data[method_name]
        df_clean = df[[x_var, "rel_error_mean", "rel_error_std"]].dropna()
        
        if len(df_clean) < 2:
            continue
        
        all_data[method_name] = df_clean
        
        # Update global ranges
        x_min = min(x_min, df_clean[x_var].min())
        x_max = max(x_max, df_clean[x_var].max())
        y_mean_min = min(y_mean_min, df_clean["rel_error_mean"].min())
        y_mean_max = max(y_mean_max, df_clean["rel_error_mean"].max())
        y_std_min = min(y_std_min, df_clean["rel_error_std"].min())
        y_std_max = max(y_std_max, df_clean["rel_error_std"].max())
    
    if len(all_data) < 2:
        continue
    
    # Add small padding to ranges
    x_range = x_max - x_min
    y_mean_range = y_mean_max - y_mean_min
    y_std_range = y_std_max - y_std_min
    
    x_min -= x_range * 0.05
    x_max += x_range * 0.05
    y_mean_min -= y_mean_range * 0.05
    y_mean_max += y_mean_range * 0.05
    y_std_min -= y_std_range * 0.05
    y_std_max += y_std_range * 0.05
    
    # Create figure with 2 rows (mean, std) and N columns (one per method)
    n_methods = len(all_data)
    fig, axes = plt.subplots(2, n_methods, figsize=(3.3 * n_methods, 4.4))
    
    # If only one method, axes will be 1D, convert to 2D
    if n_methods == 1:
        axes = axes.reshape(2, 1)
    
    # Plot mean relative error (row 0)
    for col_idx, (method_name, df_clean) in enumerate(all_data.items()):
        ax = axes[0, col_idx]
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_mean"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight best configuration
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_mean"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Set same limits for all subplots
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_mean_min, y_mean_max)
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_mean"])
        
        # Add significance test result to plot (top-right corner)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=6, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        if col_idx == 0:
            ax.set_ylabel("Mean relative error")
        ax.set_title(method_name, fontsize=9)
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis (only show label on bottom row)
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            unique_vals = sorted(df_clean[x_var].unique())
            ax.set_xticks(unique_vals)
    
    # Plot std relative error (row 1)
    for col_idx, (method_name, df_clean) in enumerate(all_data.items()):
        ax = axes[1, col_idx]
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_std"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight configuration with minimum std (associated with best mean)
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_std"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Set same limits for all subplots
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_std_min, y_std_max)
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_std"])
        
        # Add significance test result to plot (top-right corner)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=6, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        if col_idx == 0:
            ax.set_ylabel("Std dev of relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            unique_vals = sorted(df_clean[x_var].unique())
            ax.set_xticks(unique_vals)
    
    plt.tight_layout(pad=0.5)
    
    # Save figure
    filename = f"{x_var}_combined_mean_std.pdf"
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, bbox_inches="tight")
    print(f"  Saved: {filepath}")
    plt.close()

print("\n✅ All combined plots generated successfully!")


Found methods: ['PQ']

Hyperparameters with multiple methods: []

✅ All combined plots generated successfully!


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from scipy import stats
from itertools import cycle

# --- Configuration ---
CSV_FILE = "/data/cpanourg/2-hdvc/results/relerr/hp_tests/first_hp_test/deep_PQ_adc_vs_exact_eval.csv"
OUTPUT_DIR = Path("/home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Style (paper-quality minimalist) ---
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 8,
    "font.family": "serif",
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- X-axis label mapping ---
x_labels = {
    "train_size": "Training size",
    "n_subquantizers": "Number of subquantizers",
    "nbits": "Bits per subspace",
    "bits_per_vector": "Bits per vector",
}

# --- Time column labels ---
time_labels = {
    "train_time_s": "Training time (s)",
    "distance_table_time_s": "Distance table time (s)",
    "adc_time_s": "ADC time (s)",
    "encoding_time_s": "Encoding time (s)",
}

# --- Function to perform significance test ---
def perform_significance_test(x, y):
    """Perform Pearson correlation test and return p-value and sample size."""
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]
    
    if len(x_clean) < 3:
        return None, None
    
    if np.std(x_clean) == 0 or np.std(y_clean) == 0:
        return None, None
    
    try:
        corr, p_value = stats.pearsonr(x_clean, y_clean)
        return p_value, len(x_clean)
    except:
        return None, None

# --- Load data ---
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} rows from {CSV_FILE}")

# Define all hyperparameter columns for grouping (excluding the one on x-axis)
# Only n_subquantizers, nbits, and train_size are considered hyperparameters for grouping
all_hp_cols = ["n_subquantizers", "nbits", "train_size"]

# Define markers for different configurations
markers = ['o', 's', '^', 'v', 'D', 'p', '*', 'h', 'X', '<', '>', '1', '2', '3', '4']
marker_cycle = cycle(markers)

# --- Function to plot with connected lines ---
def plot_with_connections(df, x_var, y_var, y_label, filename_suffix):
    """Plot y_var vs x_var with lines connecting same hyperparameter configurations."""
    # Get all hyperparameter columns except x_var
    other_hp_cols = [col for col in all_hp_cols if col != x_var and col in df.columns]
    
    # For error plots, we need rel_error_mean to find best config
    cols_needed = [x_var, y_var] + other_hp_cols
    if "rel_error" in y_var and "rel_error_mean" in df.columns and "rel_error_mean" not in cols_needed:
        cols_needed.append("rel_error_mean")
    
    # Remove rows with NaN in x or y
    df_clean = df[cols_needed].dropna(subset=[x_var, y_var])
    
    if len(df_clean) < 2:
        print(f"  Skipping {y_var} vs {x_var}: insufficient data")
        return
    
    # Group by all other hyperparameters (to determine number of groups for figure size)
    grouped = df_clean.groupby(other_hp_cols) if other_hp_cols else None
    n_groups = len(grouped) if grouped is not None else 1
    
    # Increase figure size to accommodate legend and better differentiate plots
    # Adjust width based on number of groups (for legend space)
    fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))  # Cap at reasonable max
    fig_height = 3.5  # Taller than before
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Plot grouped data
    if other_hp_cols and grouped is not None:
        marker_iter = cycle(markers)
        
        for (group_key, group_df) in grouped:
            # Sort by x_var for line plotting
            group_df_sorted = group_df.sort_values(x_var)
            
            # Get marker for this group
            marker = next(marker_iter)
            
            # Create readable label
            label_parts = []
            for col, val in zip(other_hp_cols, group_key):
                # Format label nicely
                if col == "train_size":
                    if val >= 1000000:
                        label_parts.append(f"{col}={val/1e6:.1f}M")
                    elif val >= 1000:
                        label_parts.append(f"{col}={val/1e3:.0f}K")
                    else:
                        label_parts.append(f"{col}={val}")
                else:
                    label_parts.append(f"{col}={val}")
            label = ", ".join(label_parts)
            
            # Plot line connecting points
            ax.plot(
                group_df_sorted[x_var], group_df_sorted[y_var],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
    else:
        # No grouping needed, just plot all points
        df_sorted = df_clean.sort_values(x_var)
        ax.plot(
            df_sorted[x_var], df_sorted[y_var],
            marker='o', markersize=6, linewidth=1.2, alpha=0.7
        )
    
    # Highlight best configuration (minimum rel_error_mean for error plots)
    if "rel_error" in y_var and "rel_error_mean" in df_clean.columns:
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best[y_var],
            s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
        )
    
    # Labels and formatting
    ax.set_xlabel(x_labels.get(x_var, x_var))
    ax.set_ylabel(y_label)
    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
    
    # For train_size plots, set y-axis limits to focus on main data range
    if x_var == "train_size":
        # Lower bound: minimum error minus 0.02
        y_min = df_clean[y_var].min()
        y_lower = max(0, y_min - 0.02)  # Ensure it doesn't go below 0
        # Upper bound: fixed at 0.5 to exclude outliers
        ax.set_ylim(y_lower, 0.5)
    
    # Format x-axis
    if x_var == "train_size":
        # Use log scale for better visualization of train_size
        ax.set_xscale('log')
        # Format tick labels to be more readable (e.g., 10K, 100K, 1M)
        from matplotlib.ticker import FuncFormatter
        def format_train_size(x, pos):
            if x >= 1e6:
                return f'{x/1e6:.0f}M'
            elif x >= 1e3:
                return f'{x/1e3:.0f}K'
            else:
                return f'{x:.0f}'
        ax.xaxis.set_major_formatter(FuncFormatter(format_train_size))
        # Set major ticks at nice positions
        min_val = df_clean[x_var].min()
        max_val = df_clean[x_var].max()
        # Generate log-spaced ticks
        log_min = np.log10(min_val)
        log_max = np.log10(max_val)
        ticks = np.logspace(log_min, log_max, num=min(6, int(log_max - log_min) + 1))
        ax.set_xticks(ticks)
    elif x_var in ["nbits", "n_subquantizers"]:
        unique_vals = sorted(df_clean[x_var].unique())
        ax.set_xticks(unique_vals)
    
    # Always add legend if we have groups
    legend_placed_outside = False
    if other_hp_cols and grouped is not None and len(grouped) > 0:
        # Adjust font size based on number of groups
        legend_fontsize = max(6, min(8, 10 - len(grouped) * 0.1))
        # Place legend outside plot area if many groups, otherwise inside
        if len(grouped) > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            legend_placed_outside = True
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
    
    # Adjust padding based on whether legend is outside
    if legend_placed_outside:
        plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])  # Leave space on right for legend
    else:
        plt.tight_layout(pad=0.3)
    
    # Save figure
    filename = f"{x_var}_{filename_suffix}.pdf"
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, bbox_inches="tight")
    print(f"  Saved: {filepath}")
    plt.close()

# --- Generate plots for error metrics ---
print("\n=== Generating error plots ===")
x_vars_error = ["n_subquantizers", "nbits", "bits_per_vector", "train_size"]
for x_var in x_vars_error:
    if x_var not in df.columns:
        continue
    
    print(f"\nProcessing {x_var}...")
    
    # Mean relative error
    plot_with_connections(df, x_var, "rel_error_mean", "Mean relative error", "mean_rel_error")
    
    # Std relative error
    plot_with_connections(df, x_var, "rel_error_std", "Std dev of relative error", "std_rel_error")

# --- Special plot: nbits vs mean_rel_error, unified by train_size, connected by n_subquantizers and nbits ---
print("\n=== Generating special plot: nbits vs mean_rel_error (unified by train_size) ===")
if "nbits" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["nbits", "rel_error_mean", "train_size", "n_subquantizers"]
    df_special = df[cols_needed].dropna(subset=["nbits", "rel_error_mean"])
    
    if len(df_special) >= 2:
        # Group by n_subquantizers only (unifying across all train_size values)
        # Connect points with the same n_subquantizers as nbits varies
        # This means we only consider n_subquantizers and nbits for connections, ignoring train_size
        grouped_by_nsub = df_special.groupby("n_subquantizers")
        n_groups = len(grouped_by_nsub)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each n_subquantizers value, connect points as nbits varies (across all train_size values)
        for n_sub, nsub_group in grouped_by_nsub:
            # For each nbits value, we might have multiple points (different train_size)
            # We'll take the minimum (best) rel_error_mean across train_size values for each nbits
            nbits_grouped = nsub_group.groupby("nbits")["rel_error_mean"].min().reset_index()
            nbits_grouped = nbits_grouped.sort_values("nbits")
            
            # Get marker for this n_subquantizers value
            marker = next(marker_iter)
            
            # Label shows n_subquantizers (train_size is unified/ignored)
            label = f"n_subquantizers={n_sub}"
            
            # Plot line connecting points with this n_subquantizers value as nbits varies
            ax.plot(
                nbits_grouped["nbits"], nbits_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special.columns:
            best_idx = df_special["rel_error_mean"].idxmin()
            best = df_special.loc[best_idx]
            ax.scatter(
                best["nbits"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Bits per subspace")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        unique_vals = sorted(df_special["nbits"].unique())
        ax.set_xticks(unique_vals)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "nbits_mean_rel_error_unified_by_train_size.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Special plot: n_subquantizers vs mean_rel_error, unified by train_size, connected by n_subquantizers and nbits ---
print("\n=== Generating special plot: n_subquantizers vs mean_rel_error (unified by train_size) ===")
if "n_subquantizers" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["n_subquantizers", "rel_error_mean", "train_size", "nbits"]
    df_special2 = df[cols_needed].dropna(subset=["n_subquantizers", "rel_error_mean"])
    
    if len(df_special2) >= 2:
        # Group by nbits only (unifying across all train_size values)
        # Connect points with the same nbits as n_subquantizers varies
        # This means we only consider n_subquantizers and nbits for connections, ignoring train_size
        grouped_by_nbits = df_special2.groupby("nbits")
        n_groups = len(grouped_by_nbits)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each nbits value, connect points as n_subquantizers varies (across all train_size values)
        for nbits_val, nbits_group in grouped_by_nbits:
            # For each n_subquantizers value, we might have multiple points (different train_size)
            # We'll take the minimum (best) rel_error_mean across train_size values for each n_subquantizers
            nsub_grouped = nbits_group.groupby("n_subquantizers")["rel_error_mean"].min().reset_index()
            nsub_grouped = nsub_grouped.sort_values("n_subquantizers")
            
            # Get marker for this nbits value
            marker = next(marker_iter)
            
            # Label shows nbits (train_size is unified/ignored)
            label = f"nbits={nbits_val}"
            
            # Plot line connecting points with this nbits value as n_subquantizers varies
            ax.plot(
                nsub_grouped["n_subquantizers"], nsub_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special2.columns:
            best_idx = df_special2["rel_error_mean"].idxmin()
            best = df_special2.loc[best_idx]
            ax.scatter(
                best["n_subquantizers"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Number of subquantizers")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        unique_vals = sorted(df_special2["n_subquantizers"].unique())
        ax.set_xticks(unique_vals)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "n_subquantizers_mean_rel_error_unified_by_train_size.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Special plot: bits_per_vector vs mean_rel_error, unified by nbits and train_size, differentiated by n_subquantizers ---
print("\n=== Generating special plot: bits_per_vector vs mean_rel_error (unified by nbits and train_size) ===")
if "bits_per_vector" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["bits_per_vector", "rel_error_mean", "train_size", "nbits", "n_subquantizers"]
    df_special3 = df[cols_needed].dropna(subset=["bits_per_vector", "rel_error_mean"])
    
    if len(df_special3) >= 2:
        # Group by n_subquantizers only (unifying across all nbits and train_size values)
        # Connect points with the same n_subquantizers as bits_per_vector varies
        grouped_by_nsub = df_special3.groupby("n_subquantizers")
        n_groups = len(grouped_by_nsub)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each n_subquantizers value, connect points as bits_per_vector varies (across all nbits and train_size values)
        for n_sub, nsub_group in grouped_by_nsub:
            # For each bits_per_vector value, we might have multiple points (different nbits and train_size)
            # We'll take the minimum (best) rel_error_mean across nbits and train_size values for each bits_per_vector
            bits_grouped = nsub_group.groupby("bits_per_vector")["rel_error_mean"].min().reset_index()
            bits_grouped = bits_grouped.sort_values("bits_per_vector")
            
            # Get marker for this n_subquantizers value
            marker = next(marker_iter)
            
            # Label shows n_subquantizers (nbits and train_size are unified/ignored)
            label = f"n_subquantizers={n_sub}"
            
            # Plot line connecting points with this n_subquantizers value as bits_per_vector varies
            ax.plot(
                bits_grouped["bits_per_vector"], bits_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special3.columns:
            best_idx = df_special3["rel_error_mean"].idxmin()
            best = df_special3.loc[best_idx]
            ax.scatter(
                best["bits_per_vector"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Bits per vector")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        unique_vals = sorted(df_special3["bits_per_vector"].unique())
        ax.set_xticks(unique_vals)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "bits_per_vector_mean_rel_error_unified_by_nbits_train_size.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Special plot: bits_per_vector vs mean_rel_error, unified by n_subquantizers and train_size, differentiated by nbits ---
print("\n=== Generating special plot: bits_per_vector vs mean_rel_error (unified by n_subquantizers and train_size) ===")
if "bits_per_vector" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["bits_per_vector", "rel_error_mean", "train_size", "nbits", "n_subquantizers"]
    df_special4 = df[cols_needed].dropna(subset=["bits_per_vector", "rel_error_mean"])
    
    if len(df_special4) >= 2:
        # Group by nbits only (unifying across all n_subquantizers and train_size values)
        # Connect points with the same nbits as bits_per_vector varies
        grouped_by_nbits = df_special4.groupby("nbits")
        n_groups = len(grouped_by_nbits)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each nbits value, connect points as bits_per_vector varies (across all n_subquantizers and train_size values)
        for nbits_val, nbits_group in grouped_by_nbits:
            # For each bits_per_vector value, we might have multiple points (different n_subquantizers and train_size)
            # We'll take the minimum (best) rel_error_mean across n_subquantizers and train_size values for each bits_per_vector
            bits_grouped = nbits_group.groupby("bits_per_vector")["rel_error_mean"].min().reset_index()
            bits_grouped = bits_grouped.sort_values("bits_per_vector")
            
            # Get marker for this nbits value
            marker = next(marker_iter)
            
            # Label shows nbits (n_subquantizers and train_size are unified/ignored)
            label = f"nbits={nbits_val}"
            
            # Plot line connecting points with this nbits value as bits_per_vector varies
            ax.plot(
                bits_grouped["bits_per_vector"], bits_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special4.columns:
            best_idx = df_special4["rel_error_mean"].idxmin()
            best = df_special4.loc[best_idx]
            ax.scatter(
                best["bits_per_vector"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Bits per vector")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        unique_vals = sorted(df_special4["bits_per_vector"].unique())
        ax.set_xticks(unique_vals)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "bits_per_vector_mean_rel_error_unified_by_nsub_train_size.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Special plot: train_size vs mean_rel_error, unified by nbits, differentiated by n_subquantizers ---
print("\n=== Generating special plot: train_size vs mean_rel_error (unified by nbits) ===")
if "train_size" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["train_size", "rel_error_mean", "nbits", "n_subquantizers"]
    df_special5 = df[cols_needed].dropna(subset=["train_size", "rel_error_mean"])
    
    if len(df_special5) >= 2:
        # Group by nbits only (unifying across all n_subquantizers values)
        # Connect points with the same nbits as train_size varies
        grouped_by_nbits = df_special5.groupby("nbits")
        n_groups = len(grouped_by_nbits)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each nbits value, connect points as train_size varies (across all n_subquantizers values)
        for nbits_val, nbits_group in grouped_by_nbits:
            # For each train_size value, we might have multiple points (different n_subquantizers)
            # We'll take the minimum (best) rel_error_mean across n_subquantizers values for each train_size
            train_grouped = nbits_group.groupby("train_size")["rel_error_mean"].min().reset_index()
            train_grouped = train_grouped.sort_values("train_size")
            
            # Get marker for this nbits value
            marker = next(marker_iter)
            
            # Label shows nbits (n_subquantizers is unified/ignored)
            label = f"nbits={nbits_val}"
            
            # Plot line connecting points with this nbits value as train_size varies
            ax.plot(
                train_grouped["train_size"], train_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special5.columns:
            best_idx = df_special5["rel_error_mean"].idxmin()
            best = df_special5.loc[best_idx]
            ax.scatter(
                best["train_size"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Training size")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis with log scale
        ax.set_xscale('log')
        from matplotlib.ticker import FuncFormatter
        def format_train_size(x, pos):
            if x >= 1e6:
                return f'{x/1e6:.0f}M'
            elif x >= 1e3:
                return f'{x/1e3:.0f}K'
            else:
                return f'{x:.0f}'
        ax.xaxis.set_major_formatter(FuncFormatter(format_train_size))
        min_val = df_special5["train_size"].min()
        max_val = df_special5["train_size"].max()
        log_min = np.log10(min_val)
        log_max = np.log10(max_val)
        ticks = np.logspace(log_min, log_max, num=min(6, int(log_max - log_min) + 1))
        ax.set_xticks(ticks)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "train_size_mean_rel_error_unified_by_nbits.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Special plot: train_size vs mean_rel_error, unified by n_subquantizers, differentiated by nbits ---
print("\n=== Generating special plot: train_size vs mean_rel_error (unified by n_subquantizers) ===")
if "train_size" in df.columns and "rel_error_mean" in df.columns:
    # Get required columns
    cols_needed = ["train_size", "rel_error_mean", "nbits", "n_subquantizers"]
    df_special6 = df[cols_needed].dropna(subset=["train_size", "rel_error_mean"])
    
    if len(df_special6) >= 2:
        # Group by n_subquantizers only (unifying across all nbits values)
        # Connect points with the same n_subquantizers as train_size varies
        grouped_by_nsub = df_special6.groupby("n_subquantizers")
        n_groups = len(grouped_by_nsub)
        
        # Figure size
        fig_width = max(5.0, 4.0 + min(n_groups * 0.3, 3.0))
        fig_height = 3.5
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
        marker_iter = cycle(markers)
        
        # For each n_subquantizers value, connect points as train_size varies (across all nbits values)
        for n_sub, nsub_group in grouped_by_nsub:
            # For each train_size value, we might have multiple points (different nbits)
            # We'll take the minimum (best) rel_error_mean across nbits values for each train_size
            train_grouped = nsub_group.groupby("train_size")["rel_error_mean"].min().reset_index()
            train_grouped = train_grouped.sort_values("train_size")
            
            # Get marker for this n_subquantizers value
            marker = next(marker_iter)
            
            # Label shows n_subquantizers (nbits is unified/ignored)
            label = f"n_subquantizers={n_sub}"
            
            # Plot line connecting points with this n_subquantizers value as train_size varies
            ax.plot(
                train_grouped["train_size"], train_grouped["rel_error_mean"],
                marker=marker, markersize=6, linewidth=1.2, alpha=0.7,
                label=label
            )
        
        # Highlight best configuration
        if "rel_error_mean" in df_special6.columns:
            best_idx = df_special6["rel_error_mean"].idxmin()
            best = df_special6.loc[best_idx]
            ax.scatter(
                best["train_size"], best["rel_error_mean"],
                s=100, marker="*", color="red", edgecolors="black", zorder=5, linewidths=0.5
            )
        
        # Labels and formatting
        ax.set_xlabel("Training size")
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis with log scale
        ax.set_xscale('log')
        from matplotlib.ticker import FuncFormatter
        def format_train_size(x, pos):
            if x >= 1e6:
                return f'{x/1e6:.0f}M'
            elif x >= 1e3:
                return f'{x/1e3:.0f}K'
            else:
                return f'{x:.0f}'
        ax.xaxis.set_major_formatter(FuncFormatter(format_train_size))
        min_val = df_special6["train_size"].min()
        max_val = df_special6["train_size"].max()
        log_min = np.log10(min_val)
        log_max = np.log10(max_val)
        ticks = np.logspace(log_min, log_max, num=min(6, int(log_max - log_min) + 1))
        ax.set_xticks(ticks)
        
        # Add legend
        legend_fontsize = max(6, min(8, 10 - n_groups * 0.1))
        if n_groups > 6:
            ax.legend(fontsize=legend_fontsize, loc='center left', bbox_to_anchor=(1, 0.5), 
                     framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3, rect=[0, 0, 0.85, 1])
        else:
            ax.legend(fontsize=legend_fontsize, loc='best', framealpha=0.9, fancybox=True, shadow=True)
            plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = "train_size_mean_rel_error_unified_by_nsub.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
    else:
        print("  Skipping: insufficient data")
else:
    print("  Skipping: required columns not found")

# --- Generate plots for time metrics (only for n_subquantizers and nbits) ---
print("\n=== Generating time plots ===")
x_vars_time = ["n_subquantizers", "nbits"]
time_vars = ["train_time_s", "distance_table_time_s", "adc_time_s", "encoding_time_s"]

for x_var in x_vars_time:
    if x_var not in df.columns:
        continue
    
    print(f"\nProcessing {x_var}...")
    
    for time_var in time_vars:
        if time_var not in df.columns:
            continue
        
        # Create filename suffix
        time_suffix = time_var.replace("_time_s", "").replace("_", "_")
        filename_suffix = f"{time_suffix}_time"
        
        plot_with_connections(df, x_var, time_var, time_labels.get(time_var, time_var), filename_suffix)

print("\n✅ All plots generated successfully!")


Loaded 41 rows from /data/cpanourg/2-hdvc/results/relerr/hp_tests/first_hp_test/deep_PQ_adc_vs_exact_eval.csv

=== Generating error plots ===

Processing n_subquantizers...
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/n_subquantizers_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/n_subquantizers_std_rel_error.pdf

Processing nbits...
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/nbits_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/nbits_std_rel_error.pdf

Processing bits_per_vector...
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/bits_per_vector_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/pq/bits_per_vector_std_rel_error.pdf

Processing train_size...
  Saved: /home/c